# Download Library

In [ ]:
%pip install pandas numpy scikit-learn joblib sentence-transformers faiss-cpu regex openpyxl pyarrow fastparquet

 # Import Library

In [73]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from sentence_transformers import SentenceTransformer
import faiss
import re
from collections.abc import Iterable

# **Load Dataset**

# **Dataset Learning Path dan Course**

In [74]:
path1='./LP and Course Mapping.xlsx'
dfLPCourse = pd.read_excel(path1, sheet_name='LP + Course')
dfCourse = pd.read_excel(path1, sheet_name='Course')
dfLP = pd.read_excel(path1, sheet_name='Learning Path')
dfCourseLevel = pd.read_excel(path1, sheet_name='Course Level')
dfTutorials = pd.read_excel(path1, sheet_name='Tutorials')

## **Data Frame LP + Course**

In [75]:
dfLPCourse.head()

,learning_path_name,course_name,course_level_str,tutorial_title
0,AI Engineer,Belajar Dasar AI,Dasar,Taksonomi AI
1,AI Engineer,Belajar Dasar AI,Dasar,[Story] Machine Learning: Harapan menjadi keny...
2,AI Engineer,Belajar Dasar AI,Dasar,Rangkuman Kelas
3,AI Engineer,Belajar Dasar AI,Dasar,Tipe-Tipe Machine Learning
4,AI Engineer,Belajar Dasar AI,Dasar,Forum Diskusi


## **Data Frame Learning Path**

In [76]:
dfLP.head()

,learning_path_id,learning_path_name
0,1,AI Engineer
1,2,Android Developer
2,3,Back-End Developer JavaScript
3,4,Back-End Developer Python
4,5,Data Scientist


## **Data Frame Course**

In [77]:
dfCourse.head()

,course_id,learning_path_id,course_name,course_level_str,hours_to_study
0,1,1,Belajar Dasar AI,1,10
1,2,1,Belajar Fundamental Deep Learning,3,110
2,3,1,Belajar Machine Learning untuk Pemula,2,90
3,4,1,Machine Learning Terapan,4,80
4,5,1,Membangun Proyek Deep Learning Tingkat Mahir,4,90


## **Data Frame Course Level**

In [78]:
dfCourseLevel.head()

,id,course_level
0,1,Dasar
1,2,Pemula
2,3,Menengah
3,4,Mahir
4,5,Profesional


## **Data Frame Tutorials**

In [79]:
dfTutorials.head()

,tutorial_id,course_id,tutorial_title
0,1,1,Taksonomi AI
1,2,1,[Story] Machine Learning: Harapan menjadi keny...
2,3,1,Rangkuman Kelas
3,4,1,Tipe-Tipe Machine Learning
4,5,1,Forum Diskusi


# **Dataset Resource Learning Buddy**

In [80]:
path2='./Resource Data Learning Buddy.xlsx'
dfLPAnswer = pd.read_excel(path2, sheet_name='Learning Path Answer')
dfCIQ = pd.read_excel(path2, sheet_name='Current Interest Questions')
dfCTQ = pd.read_excel(path2, sheet_name='Current Tech Questions')
dfSkillKeywords = pd.read_excel(path2, sheet_name='Skill Keywords')
dfStudentProgress = pd.read_excel(path2, sheet_name='Student Progress')

## **Data Frame Learning Path Answer**

In [81]:
dfLPAnswer.head()

,id,name,summary,description,course_difficulty,course_price,technologies,course_type,courseMeta,courseInfo
0,14,Belajar Fundamental Aplikasi Android,Pelajari skill Android dengan kurikulum terlen...,Android merupakan sistem operasi mobile dengan...,Menengah,PAID,Android,Picodiploma,"['140 Jam', '4,84', 'Menengah']","['107 Modul', '41.831 Siswa Terdaftar']"
1,32,Belajar Membangun LINE Chatbot,Belajar membuat chatbot yang menarik pada plat...,LINE merupakan platform komunikasi yang sangat...,Pemula,PAID,"Android,Chatbot,Web",Reguler,"['20 Jam', '4,89', 'Pemula']","['26 Modul', '15.478 Siswa Terdaftar']"
2,51,Belajar Membuat Aplikasi Android untuk Pemula,Buat aplikasi pertamamu pada Android Studio de...,Android merupakan sistem operasi mobile dengan...,Pemula,PAID,Android,Reguler,"['60 Jam', '4,87', 'Pemula']","['49 Modul', '116.344 Siswa Terdaftar']"
3,60,Memulai Pemrograman Dengan Java,Belajar bahasa Java buat kamu yang ingin mempe...,Java merupakan bahasa yang diciptakan oleh Jam...,Dasar,FREE,"Android,Multi Platform",Reguler,"['15 Jam', '4,86', 'Dasar']","['42 Modul', '80.316 Siswa Terdaftar']"
4,80,Memulai Pemrograman dengan Kotlin,"Pelajari dasar bahasa pemrograman, functional ...",Kotlin merupakan bahasa utama yang digunakan d...,"Dasar,Pemula",PAID,"Android,Multi Platform",Reguler,"['50 Jam', '4,84', 'Dasar - Pemula']","['132 Modul', '59.512 Siswa Terdaftar']"


In [82]:
dfCIQ.head()

,question_desc,option_text,category
0,Mana kegiatan yang paling relate denganmu di p...,Mencoba membuat menu sarapan,Mobile Development
1,Mana kegiatan yang paling relate denganmu di p...,Baca atau lihat info viral dari berbagai sumber,Artificial Intelligence
2,Mana kegiatan yang paling relate denganmu di p...,Membersihkan kamar,Cloud Computing
3,Mana kegiatan yang paling relate denganmu di p...,Coret-coret atau menulis di buku,Web Development
4,"Jika sedang menghadapi masalah, cara mana yang...",Coba berbagai cara sampai menemukan solusinya,Mobile Development


In [83]:
dfCTQ.head()

,tech_category,difficulty,question_desc,option_1,option_2,option_3,option_4,correct_answer
0,Android,beginner,Apa yang dimaksud dengan Activity dalam pengem...,Komponen yang menangani tampilan pengguna,Kelas untuk menyimpan data,Fungsi untuk menjalankan proses di latar belakang,Antarmuka untuk menghubungkan database,Komponen yang menangani tampilan pengguna
1,Android,beginner,Apa fungsi utama dari file AndroidManifest.xml?,Menyimpan data aplikasi,Mendefinisikan struktur dan metadata aplikasi,Mengatur tata letak UI,Mengelola dependensi proyek,Mendefinisikan struktur dan metadata aplikasi
2,Android,beginner,"Dalam konteks Android, apa yang dimaksud denga...",Komponen untuk menyimpan data,Mekanisme untuk navigasi antar Activity,Fungsi untuk menangani error,Framework untuk membuat UI,Mekanisme untuk navigasi antar Activity
3,Android,beginner,Apa perbedaan utama antara LinearLayout dan Re...,"LinearLayout hanya untuk orientasi vertikal, R...","LinearLayout mengatur elemen secara berurutan,...","LinearLayout untuk aplikasi sederhana, Relativ...","Tidak ada perbedaan, keduanya dapat digunakan ...","LinearLayout mengatur elemen secara berurutan,..."
4,Android,beginner,Apa fungsi dari metode onCreate() dalam sebuah...,Menghapus Activity,Menginisialisasi Activity dan memuat layout,Menjalankan proses di background,Mengatur animasi transisi,Menginisialisasi Activity dan memuat layout


In [84]:
dfSkillKeywords.head()

,id,keyword
0,4,JavaScript
1,9,Java
2,14,Python
3,19,Kotlin
4,24,PHP


In [85]:
dfStudentProgress.head()

,name,email,course_name,active_tutorials,completed_tutorials,is_graduated,already_generated_certificate,final_submission_id,submission_rating,final_exam_id,exam_score
0,Dina Wijaya,dina.wijaya1@example.com,Belajar Membangun Aplikasi Android Native Bagi...,4,0,0,0,4.0,NaN,NaN,NaN
1,Irfan Halim,irfan.halim2@example.com,Belajar Membangun Aplikasi dengan Universal Wi...,10,0,0,0,76.0,NaN,NaN,NaN
2,Hana Pratama,hana.pratama3@example.com,Belajar Fundamental Aplikasi Android,107,1,0,0,1099.0,NaN,163.0,NaN
3,Dina Utama,dina.utama4@example.com,Belajar Fundamental Aplikasi Android,107,0,0,0,1099.0,NaN,163.0,NaN
4,Rafi Santoso,rafi.santoso5@example.com,Belajar Fundamental Aplikasi Android,107,55,0,0,1099.0,NaN,163.0,NaN


# **Missing Values**

## **Dataset Learning Path dan Course**

### **Data Frame LP + Course**

In [86]:
print("shape:", dfLPCourse.shape)
display(dfLPCourse.head(2))
print(dfLPCourse.dtypes)
print("missing:\n", dfLPCourse.isnull().sum())

shape: (7916, 4)


,learning_path_name,course_name,course_level_str,tutorial_title
0,AI Engineer,Belajar Dasar AI,Dasar,Taksonomi AI
1,AI Engineer,Belajar Dasar AI,Dasar,[Story] Machine Learning: Harapan menjadi keny...


learning_path_name    object
course_name           object
course_level_str      object
tutorial_title        object
dtype: object
missing:
 learning_path_name    0
course_name           0
course_level_str      0
tutorial_title        0
dtype: int64


### **Data Frame Learning Path**

In [87]:
print("shape:", dfLP.shape)
display(dfLP.head(2))
print(dfLP.dtypes)
print("missing:\n", dfLP.isnull().sum())

shape: (13, 2)


,learning_path_id,learning_path_name
0,1,AI Engineer
1,2,Android Developer


learning_path_id       int64
learning_path_name    object
dtype: object
missing:
 learning_path_id      0
learning_path_name    0
dtype: int64


### **Data Frame Course**

In [88]:
print("shape:", dfCourse.shape)
display(dfCourse.head(2))
print(dfCourse.dtypes)
print("missing:\n", dfCourse.isnull().sum())

shape: (75, 5)


,course_id,learning_path_id,course_name,course_level_str,hours_to_study
0,1,1,Belajar Dasar AI,1,10
1,2,1,Belajar Fundamental Deep Learning,3,110


course_id            int64
learning_path_id     int64
course_name         object
course_level_str     int64
hours_to_study       int64
dtype: object
missing:
 course_id           0
learning_path_id    0
course_name         0
course_level_str    0
hours_to_study      0
dtype: int64


### **Data Frame Course Level**

In [89]:
print("shape:", dfCourseLevel.shape)
display(dfCourseLevel.head(2))
print(dfCourseLevel.dtypes)
print("missing:\n", dfCourseLevel.isnull().sum())

shape: (5, 2)


,id,course_level
0,1,Dasar
1,2,Pemula


id               int64
course_level    object
dtype: object
missing:
 id              0
course_level    0
dtype: int64


### **Data Frame Tutorials**

In [90]:
print("shape:", dfTutorials.shape)
display(dfTutorials.head(2))
print(dfTutorials.dtypes)
print("missing:\n", dfTutorials.isnull().sum())

shape: (7771, 3)


,tutorial_id,course_id,tutorial_title
0,1,1,Taksonomi AI
1,2,1,[Story] Machine Learning: Harapan menjadi keny...


tutorial_id        int64
course_id          int64
tutorial_title    object
dtype: object
missing:
 tutorial_id       0
course_id         0
tutorial_title    0
dtype: int64


## **Dataset Resource Learning Buddy**

### **Data Frame Learning Path Answer**

In [91]:
print("shape:", dfLPAnswer.shape)
display(dfLPAnswer.head(2))
print(dfLPAnswer.dtypes)
print("missing:\n", dfLPAnswer.isnull().sum())

shape: (67, 10)


,id,name,summary,description,course_difficulty,course_price,technologies,course_type,courseMeta,courseInfo
0,14,Belajar Fundamental Aplikasi Android,Pelajari skill Android dengan kurikulum terlen...,Android merupakan sistem operasi mobile dengan...,Menengah,PAID,Android,Picodiploma,"['140 Jam', '4,84', 'Menengah']","['107 Modul', '41.831 Siswa Terdaftar']"
1,32,Belajar Membangun LINE Chatbot,Belajar membuat chatbot yang menarik pada plat...,LINE merupakan platform komunikasi yang sangat...,Pemula,PAID,"Android,Chatbot,Web",Reguler,"['20 Jam', '4,89', 'Pemula']","['26 Modul', '15.478 Siswa Terdaftar']"


id                    int64
name                 object
summary              object
description          object
course_difficulty    object
course_price         object
technologies         object
course_type          object
courseMeta           object
courseInfo           object
dtype: object
missing:
 id                   0
name                 0
summary              0
description          0
course_difficulty    0
course_price         0
technologies         0
course_type          0
courseMeta           0
courseInfo           0
dtype: int64


### **Data Frame Current Interest Question**

In [92]:
print("shape:", dfCIQ.shape)
display(dfCIQ.head(2))
print(dfCIQ.dtypes)
print("missing:\n", dfCIQ.isnull().sum())

shape: (20, 3)


,question_desc,option_text,category
0,Mana kegiatan yang paling relate denganmu di p...,Mencoba membuat menu sarapan,Mobile Development
1,Mana kegiatan yang paling relate denganmu di p...,Baca atau lihat info viral dari berbagai sumber,Artificial Intelligence


question_desc    object
option_text      object
category         object
dtype: object
missing:
 question_desc    0
option_text      0
category         0
dtype: int64


### **Data Frame Current Tech Question**

In [93]:
print("shape:", dfCTQ.shape)
display(dfCTQ.head(2))
print(dfCTQ.dtypes)
print("missing:\n", dfCTQ.isnull().sum())

shape: (326, 8)


,tech_category,difficulty,question_desc,option_1,option_2,option_3,option_4,correct_answer
0,Android,beginner,Apa yang dimaksud dengan Activity dalam pengem...,Komponen yang menangani tampilan pengguna,Kelas untuk menyimpan data,Fungsi untuk menjalankan proses di latar belakang,Antarmuka untuk menghubungkan database,Komponen yang menangani tampilan pengguna
1,Android,beginner,Apa fungsi utama dari file AndroidManifest.xml?,Menyimpan data aplikasi,Mendefinisikan struktur dan metadata aplikasi,Mengatur tata letak UI,Mengelola dependensi proyek,Mendefinisikan struktur dan metadata aplikasi


tech_category     object
difficulty        object
question_desc     object
option_1          object
option_2          object
option_3          object
option_4          object
correct_answer    object
dtype: object
missing:
 tech_category     0
difficulty        0
question_desc     0
option_1          0
option_2          0
option_3          0
option_4          0
correct_answer    0
dtype: int64


### **Data Frame Skill Keywords**

In [94]:
print("shape:", dfSkillKeywords.shape)
display(dfSkillKeywords.head(2))
print(dfSkillKeywords.dtypes)
print("missing:\n", dfSkillKeywords.isnull().sum())

shape: (5553, 2)


,id,keyword
0,4,JavaScript
1,9,Java


id          int64
keyword    object
dtype: object
missing:
 id         0
keyword    1
dtype: int64


### **Data Frame Student Progress**

In [95]:
print("shape:", dfStudentProgress.shape)
display(dfStudentProgress.head(2))
print(dfStudentProgress.dtypes)
print("missing:\n", dfStudentProgress.isnull().sum())

shape: (271, 11)


,name,email,course_name,active_tutorials,completed_tutorials,is_graduated,already_generated_certificate,final_submission_id,submission_rating,final_exam_id,exam_score
0,Dina Wijaya,dina.wijaya1@example.com,Belajar Membangun Aplikasi Android Native Bagi...,4,0,0,0,4.0,NaN,NaN,NaN
1,Irfan Halim,irfan.halim2@example.com,Belajar Membangun Aplikasi dengan Universal Wi...,10,0,0,0,76.0,NaN,NaN,NaN


name                              object
email                             object
course_name                       object
active_tutorials                   int64
completed_tutorials                int64
is_graduated                       int64
already_generated_certificate      int64
final_submission_id              float64
submission_rating                float64
final_exam_id                    float64
exam_score                       float64
dtype: object
missing:
 name                               0
email                              0
course_name                        0
active_tutorials                   0
completed_tutorials                0
is_graduated                       0
already_generated_certificate      0
final_submission_id              139
submission_rating                218
final_exam_id                     17
exam_score                       144
dtype: int64


# **Duplication**

In [96]:
dataFrames = {
    'LP + Course': dfLPCourse,
    'Learning Path': dfLP,
    'Course': dfCourse,
    'Course Level': dfCourseLevel,
    'Tutorials': dfTutorials,
    'Learning Path Answer': dfLPAnswer,
    'Current Interest Questions': dfCIQ,
    'Current Tech Questions': dfCTQ,
    'Skill Keywords': dfSkillKeywords,
    'Student Progress': dfStudentProgress
}

for name, df in dataFrames.items():
    dupCount = df.duplicated().sum()
    print(f"{name}: duplicate rows = {dupCount}")

if dfCourse.duplicated().sum() > 0:
    display(dfCourse[dfCourse.duplicated(keep=False)].head(10))

LP + Course: duplicate rows = 145
Learning Path: duplicate rows = 0
Course: duplicate rows = 0
Course Level: duplicate rows = 0
Tutorials: duplicate rows = 0
Learning Path Answer: duplicate rows = 0
Current Interest Questions: duplicate rows = 0
Current Tech Questions: duplicate rows = 0
Skill Keywords: duplicate rows = 0
Student Progress: duplicate rows = 0


In [97]:
dupeLPCourse = dfLPCourse[dfLPCourse.duplicated(keep=False)]
print("Jumlah baris duplikat:", dupeLPCourse.shape[0])
display(dupeLPCourse)

Jumlah baris duplikat: 214


,learning_path_name,course_name,course_level_str,tutorial_title
314,AI Engineer,Machine Learning Terapan,Mahir,Analisis Sentimen dengan Deep Learning
326,AI Engineer,Machine Learning Terapan,Mahir,Detail Laporan
355,AI Engineer,Machine Learning Terapan,Mahir,Detail Laporan
359,AI Engineer,Machine Learning Terapan,Mahir,Model Development
368,AI Engineer,Machine Learning Terapan,Mahir,Data Preparation
...,...,...,...,...
7384,React Developer,Belajar Dasar Pemrograman JavaScript,Dasar,Rangkuman Materi
7470,React Developer,Belajar Dasar Pemrograman Web,Dasar,Rangkuman Pengenalan HTML
7474,React Developer,Belajar Dasar Pemrograman Web,Dasar,Rangkuman Pengenalan CSS
7526,React Developer,Belajar Dasar Pemrograman Web,Dasar,Rangkuman Pengenalan HTML


## **Data LP + Course untuk Roadmap**

In [98]:
dfLpCourseUnique = dfLPCourse[['learning_path_name', 'course_name', 'course_level_str']].drop_duplicates().reset_index(drop=True)
dfLpCourseUnique.head()

,learning_path_name,course_name,course_level_str
0,AI Engineer,Belajar Dasar AI,Dasar
1,AI Engineer,Belajar Fundamental Deep Learning,Menengah
2,AI Engineer,Belajar Machine Learning untuk Pemula,Pemula
3,AI Engineer,Machine Learning Terapan,Mahir
4,AI Engineer,Membangun Proyek Deep Learning Tingkat Mahir,Mahir


## **Data Tutorials per course**

In [99]:
idToName = dict(zip(dfCourse['course_id'], dfCourse['course_name']))

dfTutorialsPerCourse = (
    dfTutorials.groupby('course_id')['tutorial_title']
    .apply(list)
    .reset_index()
)

dfTutorialsPerCourse['course_name'] = dfTutorialsPerCourse['course_id'].map(idToName)
dfTutorialsPerCourse = dfTutorialsPerCourse[['course_name', 'tutorial_title']]

REMOVE_BRACKET_TAGS = True
NOISE_KEYWORDS = ['deprecated', 'depracat', 'glossarium', 'glossary', 'knowledge check', 'quiz', 'kuis', 'exercise', 'latihan']
MAX_SHOW = 25

def remove_bracket_tags(s: str) -> str:
    return re.sub(r'\[.*?\]\s*', '', s).strip()

def normalize_item(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    s = s.strip()
    if REMOVE_BRACKET_TAGS:
        s = remove_bracket_tags(s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'^[\:\-\–\—\.\,]+\s*', '', s)
    return s.strip()

def is_noise(s: str) -> bool:
    if not s:
        return True
    low = s.lower()
    if len(low) <= 2:
        return True
    for kw in NOISE_KEYWORDS:
        if kw in low:
            return True
    if re.fullmatch(r'[\W_]+', s):
        return True
    return False

def dedupe_preserve_order(lst):
    out = []
    seen = set()
    for it in lst:
        if it not in seen:
            out.append(it)
            seen.add(it)
    return out

cleaned_full = []
cleaned_kb = []
clean_removed_summary = {}

for _, r in dfTutorialsPerCourse.iterrows():
    raw = r['tutorial_title']
    if raw is None:
        items = []
    elif isinstance(raw, str):
        items = [raw]
    elif isinstance(raw, Iterable):
        items = list(raw)
    else:
        items = [str(raw)]

    cleaned = []
    removed_count = 0
    for it in items:
        if it is None:
            continue
        s = normalize_item(str(it))
        if is_noise(s):
            removed_count += 1
            clean_removed_summary[s] = clean_removed_summary.get(s, 0) + 1
            continue
        if s == "":
            continue
        cleaned.append(s)
    cleaned = dedupe_preserve_order(cleaned)
    cleaned_full.append(cleaned)
    cleaned_kb.append(cleaned[:MAX_SHOW])

dfTutorialsPerCourse['tutorial_list_clean'] = cleaned_full
dfTutorialsPerCourse['tutorial_list_kb'] = cleaned_kb
dfTutorialsPerCourse['jumlah_tutorial'] = dfTutorialsPerCourse['tutorial_list_clean'].apply(len)

print("Cleaning done. Courses processed:", len(dfTutorialsPerCourse))
total_removed_examples = sum(clean_removed_summary.values())
print("Sample removed item counts (top 10):")
for k, v in sorted(clean_removed_summary.items(), key=lambda x: -x[1])[:10]:
    print(f"  {k!r}: {v}")

display(dfTutorialsPerCourse.head())
kb_preview = []
for _, r in dfTutorialsPerCourse.head(20).iterrows():
    kb_preview.append({'course_name': r['course_name'],
                       'modules_preview': "; ".join(r['tutorial_list_kb'])})
display(pd.DataFrame(kb_preview))

Cleaning done. Courses processed: 75
Sample removed item counts (top 10):
  'Glossarium': 17
  '': 12
  'Kuis Control Flow': 6
  'Kuis Coding : Array': 6
  'Kuis Coding: Fungsi': 5
  'Kuis Coding: Operasi List': 5
  'Kuis Coding: Ekspresi': 5
  'Kuis Object-Oriented Programming (OOP)': 5
  'Kuis Coding: Perulangan dan Percabangan': 5
  'Kuis Berkenalan dengan Python': 5


,course_name,tutorial_title,tutorial_list_clean,tutorial_list_kb,jumlah_tutorial
0,Belajar Dasar AI,"[Taksonomi AI, [Story] Machine Learning: Harap...","[Taksonomi AI, Machine Learning: Harapan menja...","[Taksonomi AI, Machine Learning: Harapan menja...",34
1,Belajar Fundamental Deep Learning,"[Glossarium, Pra-pemrosesan Data Gambar, Penga...","[Pra-pemrosesan Data Gambar, Pengantar Deep Le...","[Pra-pemrosesan Data Gambar, Pengantar Deep Le...",114
2,Belajar Machine Learning untuk Pemula,"[Kuis Hi, Machine Learning!, Latihan Studi Kas...","[Decision Tree, Tugas Akhir Kuliah, Pengenalan...","[Decision Tree, Tugas Akhir Kuliah, Pengenalan...",115
3,Machine Learning Terapan,"[Pengenalan Computer Vision, Analisis Sentimen...","[Pengenalan Computer Vision, Analisis Sentimen...","[Pengenalan Computer Vision, Analisis Sentimen...",99
4,Membangun Proyek Deep Learning Tingkat Mahir,"[Reproducibility dalam TensorFlow, Mengurutkan...","[Reproducibility dalam TensorFlow, Mengurutkan...","[Reproducibility dalam TensorFlow, Mengurutkan...",69


,course_name,modules_preview
0,Belajar Dasar AI,Taksonomi AI; Machine Learning: Harapan menjad...
1,Belajar Fundamental Deep Learning,Pra-pemrosesan Data Gambar; Pengantar Deep Lea...
2,Belajar Machine Learning untuk Pemula,Decision Tree; Tugas Akhir Kuliah; Pengenalan ...
3,Machine Learning Terapan,Pengenalan Computer Vision; Analisis Sentimen ...
4,Membangun Proyek Deep Learning Tingkat Mahir,Reproducibility dalam TensorFlow; Mengurutkan ...
5,Memulai Pemrograman dengan Python,Library Text Processing; Bersiap Membuat Kode ...
6,Belajar Fundamental Aplikasi Android,Teori AlarmManager; Ringkasan View dan ViewGro...
7,Belajar Membuat Aplikasi Android untuk Pemula,Pengenalan Project pada Android Studio; Debugg...
8,Belajar Pengembangan Aplikasi Android Intermed...,Teori WebView; Teori Localization pada Teks; T...
9,Belajar Prinsip Pemrograman SOLID,Generalization dan Specialization; Studi Kasus...


In [100]:
dfTutorialsPerCourse.head()

,course_name,tutorial_title,tutorial_list_clean,tutorial_list_kb,jumlah_tutorial
0,Belajar Dasar AI,"[Taksonomi AI, [Story] Machine Learning: Harap...","[Taksonomi AI, Machine Learning: Harapan menja...","[Taksonomi AI, Machine Learning: Harapan menja...",34
1,Belajar Fundamental Deep Learning,"[Glossarium, Pra-pemrosesan Data Gambar, Penga...","[Pra-pemrosesan Data Gambar, Pengantar Deep Le...","[Pra-pemrosesan Data Gambar, Pengantar Deep Le...",114
2,Belajar Machine Learning untuk Pemula,"[Kuis Hi, Machine Learning!, Latihan Studi Kas...","[Decision Tree, Tugas Akhir Kuliah, Pengenalan...","[Decision Tree, Tugas Akhir Kuliah, Pengenalan...",115
3,Machine Learning Terapan,"[Pengenalan Computer Vision, Analisis Sentimen...","[Pengenalan Computer Vision, Analisis Sentimen...","[Pengenalan Computer Vision, Analisis Sentimen...",99
4,Membangun Proyek Deep Learning Tingkat Mahir,"[Reproducibility dalam TensorFlow, Mengurutkan...","[Reproducibility dalam TensorFlow, Mengurutkan...","[Reproducibility dalam TensorFlow, Mengurutkan...",69


# **Knowledge Base**

In [101]:
ARTIFACTS_DIR = 'app/artifacts'

def detect_lpanswer_cols(df):
    name_col = None
    summary_col = None
    for c in df.columns:
        if c.lower() in ('name','course_name','title'):
            name_col = c
            break
    for c in df.columns:
        if c.lower() in ('summary','short_description','overview'):
            summary_col = c
            break
    if summary_col is None:
        for c in df.columns:
            if 'description' in c.lower():
                summary_col = c
                break
    if name_col is None:
        text_cols = [c for c in df.columns if df[c].dtype == object]
        name_col = text_cols[0] if text_cols else df.columns[0]
    return name_col, summary_col

nameCol, summaryCol = detect_lpanswer_cols(dfLPAnswer)

courseSummaryMap = {}
for _, r in dfLPAnswer.iterrows():
    key = r.get(nameCol)
    if pd.isna(key):
        continue
    key = str(key).strip()
    if summaryCol and pd.notna(r.get(summaryCol)):
        courseSummaryMap[key] = str(r.get(summaryCol)).strip()
    else:
        courseSummaryMap[key] = ""

tutorials_kb = {}
tutorials_full = {}
if 'course_name' in dfTutorialsPerCourse.columns:
    for _, r in dfTutorialsPerCourse.iterrows():
        cname = r.get('course_name')
        if pd.isna(cname):
            continue
        cname = str(cname).strip()
        kb_list = r.get('tutorial_list_kb', [])
        full_list = r.get('tutorial_list_clean', [])
        tutorials_kb[cname] = kb_list if isinstance(kb_list, list) else ([] if pd.isna(kb_list) else [str(kb_list)])
        tutorials_full[cname] = full_list if isinstance(full_list, list) else ([] if pd.isna(full_list) else [str(full_list)])
else:
    id_to_name = dict(zip(dfCourse['course_id'], dfCourse['course_name'])) if 'course_id' in dfCourse.columns else {}
    for _, r in dfTutorialsPerCourse.iterrows():
        key = r.get('course_id', None)
        cname = id_to_name.get(key, str(key))
        kb_list = r.get('tutorial_list_kb', [])
        full_list = r.get('tutorial_list_clean', [])
        tutorials_kb[cname] = kb_list if isinstance(kb_list, list) else ([] if pd.isna(kb_list) else [str(kb_list)])
        tutorials_full[cname] = full_list if isinstance(full_list, list) else ([] if pd.isna(full_list) else [str(full_list)])

kb_course = []
for _, row in dfCourse.iterrows():
    title = row.get('course_name') if 'course_name' in dfCourse.columns else (row.get('course_id') if 'course_id' in dfCourse.columns else '')
    if pd.isna(title):
        continue
    title = str(title).strip()
    summary = courseSummaryMap.get(title, "")
    parts = []
    if summary:
        parts.append(summary)
    if 'course_level_str' in dfCourse.columns and pd.notna(row.get('course_level_str')):
        parts.append(f"Level: {row.get('course_level_str')}.")
    if 'hours_to_study' in dfCourse.columns and pd.notna(row.get('hours_to_study')):
        parts.append(f"Estimated hours: {row.get('hours_to_study')}.")
    modules = tutorials_kb.get(title, [])
    if modules:
        parts.append("Modules: " + "; ".join(modules))
    if not parts:
        parts = [f"Course: {title}"]
    kb_course.append({'title': title, 'text': " ".join(parts), 'type': 'course'})

kbCourseFinal = pd.DataFrame(kb_course)

kb_tutorials = []
for cname, tlist in tutorials_kb.items():
    if not cname:
        continue
    text = "; ".join(tlist) if isinstance(tlist, (list,tuple)) else str(tlist)
    if text.strip():
        kb_tutorials.append({'title': cname, 'text': text, 'type': 'tutorials'})

kbTutorialsFinal = pd.DataFrame(kb_tutorials)

kb_roadmap = []
if 'learning_path_name' in dfLpCourseUnique.columns and 'course_name' in dfLpCourseUnique.columns:
    grp = dfLpCourseUnique.groupby('learning_path_name')['course_name'].apply(list).reset_index()
    for _, r in grp.iterrows():
        title = r['learning_path_name']
        lst = r['course_name'] if isinstance(r['course_name'], list) else [r['course_name']]
        text = "; ".join(lst)
        kb_roadmap.append({'title': title, 'text': text, 'type': 'roadmap'})

kbRoadmapFinal = pd.DataFrame(kb_roadmap)

kb_lp = []
if 'learning_path_name' in dfLpCourseUnique.columns:
    grp_count = dfLpCourseUnique.groupby('learning_path_name')['course_name'].nunique().reset_index(name='n_courses')
    for _, r in grp_count.iterrows():
        title = r['learning_path_name']
        n = int(r['n_courses'])
        text = f"Learning path {title} terdiri dari {n} course yang disusun untuk membantu kamu mencapai kompetensi terkait."
        kb_lp.append({'title': title, 'text': text, 'type': 'learning_path'})

kbLearningPathFinal = pd.DataFrame(kb_lp)

kb_final = pd.concat([kbCourseFinal, kbLearningPathFinal, kbTutorialsFinal, kbRoadmapFinal], ignore_index=True, sort=False)
kb_final['text'] = kb_final['text'].fillna('').astype(str)
kb_final = kb_final[kb_final['text'].str.strip() != ""].reset_index(drop=True)
kb_final = kb_final.drop_duplicates(subset=['title','text','type']).reset_index(drop=True)

os.makedirs('data', exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

kb_final.to_csv('data/kb_final.csv', index=False)


kb_final.to_parquet(f'{ARTIFACTS_DIR}/kb.parquet', index=False, engine='fastparquet')

print("KB final saved to data/kb_final.csv and artifacts/kb.parquet")
print("Shapes:")
print(" - courses:", kbCourseFinal.shape)
print(" - learning_paths:", kbLearningPathFinal.shape)
print(" - tutorials:", kbTutorialsFinal.shape)
print(" - roadmap:", kbRoadmapFinal.shape)
print(" - kb_final total:", kb_final.shape)


kb_final.head(30)

KB final saved to data/kb_final.csv and artifacts/kb.parquet
Shapes:
 - courses: (75, 3)
 - learning_paths: (13, 3)
 - tutorials: (54, 3)
 - roadmap: (13, 3)
 - kb_final total: (134, 3)


,title,text,type
0,Belajar Dasar AI,Kelas ini memberikan pemahaman terkait dasar-d...,course
1,Belajar Fundamental Deep Learning,Level: 3. Estimated hours: 110. Modules: Semak...,course
2,Belajar Machine Learning untuk Pemula,Pelajari materi dasar pengembangan machine lea...,course
3,Machine Learning Terapan,Pelajari penerapan machine learning dengan rea...,course
4,Membangun Proyek Deep Learning Tingkat Mahir,Level: 4. Estimated hours: 90. Modules: Reprod...,course
5,Memulai Pemrograman dengan Python,Pelajari dasar pemrograman Python hingga libra...,course
6,Belajar Fundamental Aplikasi Android,Pelajari skill Android dengan kurikulum terlen...,course
7,Belajar Membuat Aplikasi Android untuk Pemula,Buat aplikasi pertamamu pada Android Studio de...,course
8,Belajar Pengembangan Aplikasi Android Intermed...,Tingkatkan pengalaman pengguna dengan mempelaj...,course
9,Belajar Prinsip Pemrograman SOLID,Pelajari kelima prinsip desain yang merupakan ...,course


# **Embeddings**

In [102]:
embModel = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

ARTIFACTS_DIR = 'app/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print("Encoding KB...")
kb_final['embedding'] = kb_final['text'].apply(lambda x: embModel.encode(str(x)))

embedding_matrix = np.vstack(kb_final['embedding'].values)

np.save(f"{ARTIFACTS_DIR}/kb_embeddings.npy", embedding_matrix)

print("kb_embeddings.npy saved!")
print("Embedding shape:", embedding_matrix.shape)

Encoding KB...
kb_embeddings.npy saved!
Embedding shape: (134, 384)


# **Index FAISS**

In [103]:
ARTIFACTS_DIR = "app/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

embedding_matrix = np.load(f"{ARTIFACTS_DIR}/kb_embeddings.npy")

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("FAISS index built. Total vectors:", index.ntotal)

faiss_path = f"{ARTIFACTS_DIR}/kb_index.faiss"
faiss.write_index(index, faiss_path)

print("Saved:", faiss_path)

FAISS index built. Total vectors: 134
Saved: app/artifacts/kb_index.faiss


# **Rule-based Intent Classification**

In [104]:
class IntentClassifier:
    def __call__(self, query: str):
        q = query.lower()
        intent = {
            "intent": "default",
            "mode": "default",
            "typePriority": ["course", "learning_path", "roadmap", "tutorials"]
        }

        if any(kw in q for kw in ["berapa lama", "durasi", "butuh waktu", "jam belajar"]):
            intent.update({"intent": "duration", "mode": "duration", "typePriority": ["course"]})
        elif "learning path" in q or "lp " in q:
            intent.update({"intent": "learning_path", "typePriority": ["learning_path", "roadmap", "course"]})
        elif "modul" in q:
            intent.update({"intent": "tutorials", "typePriority": ["tutorials", "course"]})
        elif any(kw in q for kw in ["course", "kelas", "belajar", "materi"]):
            intent.update({"intent": "course", "typePriority": ["course", "roadmap", "learning_path"]})

        return intent


intent_pipe = IntentClassifier()
joblib.dump(intent_pipe, f"{ARTIFACTS_DIR}/intent_pipe.joblib")

['app/artifacts/intent_pipe.joblib']

# **Retrieval + Reranking**

In [105]:
try:
    kb_final = pd.read_parquet(f"{ARTIFACTS_DIR}/kb.parquet")
except:
    kb_final = pd.read_csv("data/kb_final.csv")

embModel = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding_matrix = np.load(f"{ARTIFACTS_DIR}/kb_embeddings.npy")

index = faiss.read_index(f"{ARTIFACTS_DIR}/kb_index.faiss")

intent_pipe = joblib.load(f"{ARTIFACTS_DIR}/intent_pipe.joblib")

print("Artifacts loaded:")
print("KB:", kb_final.shape)
print("Embeddings:", embedding_matrix.shape)
print("FAISS index total vectors:", index.ntotal)


def retrieve(query, topK=5, faissK=30):
    """
    Hybrid retrieval:
    - Encode query
    - FAISS nearest neighbors
    - Scoring (distance + typeBonus + titleBonus + durationBonus)
    """

    intent = intent_pipe(query)

    queryVec = embModel.encode([query])

    distances, indices = index.search(queryVec, faissK)

    candidates = kb_final.iloc[indices[0]].copy()
    candidates["distance"] = distances[0]
    candidates["baseScore"] = -candidates["distance"]
    typePriority = intent.get("typePriority")
    if typePriority:
        priorityMap = {t: (len(typePriority) - i) for i, t in enumerate(typePriority)}
        candidates["typeBonus"] = candidates["type"].map(priorityMap).fillna(0)
    else:
        candidates["typeBonus"] = 0

    qWords = set(re.findall(r"\w+", query.lower()))

    def keywordBonus(title):
        titleWords = set(re.findall(r"\w+", str(title).lower()))
        overlap = len(qWords & titleWords)
        return overlap * 0.5

    candidates["titleBonus"] = candidates["title"].apply(keywordBonus)

    if intent["mode"] == "duration":
        maskCourse = candidates["type"] == "course"
        if maskCourse.any():
            candidates = candidates[maskCourse].copy()

        candidates["durationBonus"] = (
            candidates["text"]
            .str.contains("Estimated hours", case=False, na=False)
            .astype(int) * 1.0
        )
    else:
        candidates["durationBonus"] = 0.0

    candidates["finalScore"] = (
        candidates["baseScore"]
        + candidates["typeBonus"]
        + candidates["titleBonus"]
        + candidates["durationBonus"]
    )

    candidates = candidates.sort_values("finalScore", ascending=False).head(topK)

    return candidates[["title", "type", "text", "finalScore"]]

print("\nTest retrieve() →")
print(retrieve("belajar web untuk pemula", topK=5))


Artifacts loaded:
KB: (134, 3)
Embeddings: (134, 384)
FAISS index total vectors: 134

Test retrieve() →
                                                title    type  \
36         Belajar Membuat Front-End Web untuk Pemula  course   
39  Belajar Membuat Aplikasi Back-End untuk Pemula...  course   
35      Belajar Fundamental Front-End Web Development  course   
37              Belajar Pengembangan Web Intermediate  course   
13          Belajar Back-End Pemula dengan JavaScript  course   

                                                 text  finalScore  
36  Pelajari materi mengenai DOM manipulation, Eve...    4.000244  
39  Belajar membuat RESTful API, dari HTTP server,...    3.451976  
35  Pelajari sintaks ES6, Web Component, dan Build...    3.098983  
37  Level: 4. Estimated hours: 80. Modules: Implem...    3.008166  
13  Level: 2. Estimated hours: 50. Modules: Membua...    2.974283  
